In [ ]:
import marimo as mo
import pandas as pd
import numpy as np
from pathlib import Path
from PIL import Image
import matplotlib.pyplot as plt
import torchvision.transforms as T
import torch

app = mo.App(width='full')

In [ ]:
def latest_run(base: Path):
    runs = [p for p in base.iterdir() if p.is_dir()]
    if not runs:
        return None
    return sorted(runs)[-1]

def list_sample_images(run_dir: Path, dataset_name: str, combo_name: str):
    folder = run_dir / 'images' / dataset_name / combo_name
    if not folder.exists():
        return []
    files = [p for p in sorted(folder.glob('sample_*.png')) if 'grid' not in p.name]
    return files

def load_img(path: Path):
    return np.asarray(Image.open(path).convert('RGB'))

def apply_combo_preview(img_np, combo):
    x = T.ToTensor()(Image.fromarray(img_np)).clamp(0, 1)
    ops = [] if combo == 'none' else combo.split('__')
    if 'add_random_noise' in ops:
        x = (x + 0.08 * torch.randn_like(x)).clamp(0, 1)
    if 'low_pass_gaussian' in ops:
        x = T.GaussianBlur(9, sigma=2.0)(x).clamp(0, 1)
    if 'high_pass_gaussian' in ops:
        low = T.GaussianBlur(9, sigma=2.0)(x)
        x = (x - low + 0.5).clamp(0, 1)
    if 'rainbow_border' in ops:
        bw = 4
        h, w = x.shape[1], x.shape[2]
        xx = torch.linspace(0, 1, w)
        yy = torch.linspace(0, 1, h)
        top = torch.stack([xx, torch.flip(xx, dims=[0]), torch.ones_like(xx)], dim=0)
        left = torch.stack([torch.ones_like(yy), yy, torch.flip(yy, dims=[0])], dim=0)
        x[:, :bw, :] = top[:, None, :].repeat(1, bw, 1)
        x[:, -bw:, :] = torch.flip(top, dims=[1])[:, None, :].repeat(1, bw, 1)
        x[:, :, :bw] = left[:, :, None].repeat(1, 1, bw)
        x[:, :, -bw:] = torch.flip(left, dims=[1])[:, :, None].repeat(1, 1, bw)
    return (x.permute(1,2,0).numpy() * 255.0).astype(np.uint8)

In [ ]:
base_out = Path('./outputs/cat_manipulation_pr2nn')
run_widget = mo.ui.text(value='', label='Run folder (optional). Leave blank = latest.')
mo.vstack([
    mo.md('### Cat Manipulation Comparison UI'),
    mo.md('Run `cat_manipulation_pr2nn.py` first, then select results here.'),
    run_widget,
])

In [ ]:
run_dir = Path(run_widget.value) if run_widget.value.strip() else latest_run(base_out)
if run_dir is None:
    raise RuntimeError('No run found in outputs/cat_manipulation_pr2nn')
metrics_path = run_dir / 'metrics.csv'
metrics = pd.read_csv(metrics_path)
combo_options = sorted(metrics['combo_name'].unique().tolist())
datasets = ['natural_cat', 'synthetic_cat_peng2023robust', 'random_noise']
run_dir, metrics.head()

In [ ]:
combo_widget = mo.ui.dropdown(options=combo_options, value=combo_options[0], label='Manipulation combination')
sample_widget = mo.ui.slider(start=0, stop=19, step=1, value=0, label='Sample index')
mo.hstack([combo_widget, sample_widget], justify='start', gap=2)

In [ ]:
combo = combo_widget.value
sample_idx = int(sample_widget.value)

metric_subset = metrics[metrics['combo_name'] == combo].copy()
metric_subset = metric_subset.set_index('dataset_name').reindex(datasets)

nat_files = list_sample_images(run_dir, 'natural_cat', combo)
syn_files = list_sample_images(run_dir, 'synthetic_cat_peng2023robust', combo)
noi_files = list_sample_images(run_dir, 'random_noise', combo)
nat_base_files = list_sample_images(run_dir, 'natural_cat', 'none')

if not nat_base_files:
    raise RuntimeError('Missing natural_cat/none image set in run output')

idx_nat = min(sample_idx, len(nat_files) - 1) if nat_files else 0
idx_syn = min(sample_idx, len(syn_files) - 1) if syn_files else 0
idx_noi = min(sample_idx, len(noi_files) - 1) if noi_files else 0
idx_base = min(sample_idx, len(nat_base_files) - 1)

img_nat_mod = load_img(nat_files[idx_nat]) if nat_files else load_img(nat_base_files[idx_base])
img_nat_base = load_img(nat_base_files[idx_base])
img_syn = load_img(syn_files[idx_syn]) if syn_files else img_nat_base
img_noi = load_img(noi_files[idx_noi]) if noi_files else img_nat_base

fig, axes = plt.subplots(2, 4, figsize=(18, 8))
axes[0,0].imshow(img_nat_base); axes[0,0].set_title('Natural cat (baseline)'); axes[0,0].axis('off')
axes[0,1].imshow(img_nat_mod); axes[0,1].set_title(f'Natural + {combo}'); axes[0,1].axis('off')
axes[0,2].imshow(img_syn); axes[0,2].set_title('Synthetic cat (Peng)'); axes[0,2].axis('off')
axes[0,3].imshow(img_noi); axes[0,3].set_title('Random noise reference'); axes[0,3].axis('off')

labels = datasets
pr_vals = metric_subset['pr'].to_numpy(dtype=float)
nn_vals = metric_subset['two_nn_mle'].to_numpy(dtype=float)

x = np.arange(len(labels))
axes[1,0].bar(x, pr_vals, color=['#4c78a8','#f58518','#54a24b'])
axes[1,0].set_xticks(x, labels, rotation=15)
axes[1,0].set_title('PR comparison')

axes[1,1].bar(x, nn_vals, color=['#4c78a8','#f58518','#54a24b'])
axes[1,1].set_xticks(x, labels, rotation=15)
axes[1,1].set_title('2NN comparison')

axes[1,2].axis('off')
axes[1,3].axis('off')
summary = metric_subset[['pr','two_nn_mle']].copy()
summary_text = '\n'.join([f"{idx}: PR={row['pr']:.3f}, 2NN={row['two_nn_mle']:.3f}" for idx, row in summary.iterrows()])
axes[1,2].text(0.02, 0.95, summary_text, va='top', family='monospace', fontsize=10)
axes[1,2].set_title('Selected combo metrics')

fig.tight_layout()
mo.vstack([
    mo.md(f'### Run: `{run_dir}`'),
    mo.md(f'### Combo: `{combo}` | Sample index: `{sample_idx}`'),
    fig,
])

In [ ]:
mo.md('Tip: use the combo dropdown as a checklist-equivalent (all precomputed combinations are available).')